![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `mistralai/mistral-small-3-1-24b-instruct-2503` to generate code based on instruction

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support for code generating in watsonx. It introduces commands for defining prompt and model testing.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to generate code using `mistralai/mistral-small-3-1-24b-instruct-2503` watsonx model based on instruction provided by the user.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
3. [Generate code based on instruction](#Generate-code-based-on-instruction)
4. [Generated code testing](#Generated-code-testing)
5. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install dependencies

In [1]:
%pip install -U ibm-watsonx-ai | tail -n 1

### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project ID
The Foundation Model requires project ID that provides the context for the call. We will obtain the ID from the project in which this notebook runs. Otherwise, please provide the project ID.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

### API Client initialization

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id=project_id)

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

In [5]:
client.foundation_models.ChatModels.show()

{'GRANITE_4_H_SMALL': 'ibm/granite-4-h-small', 'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8': 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3-1-24b-instruct-2503', 'GPT_OSS_120B': 'openai/gpt-oss-120b'}


You need to specify `model_id` that will be used for inferencing:

In [6]:
model_id = client.foundation_models.ChatModels.MISTRAL_SMALL_3_1_24B_INSTRUCT_2503

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [7]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {GenParams.TEMPERATURE: 0, GenParams.MAX_COMPLETION_TOKENS: 1024}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [8]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, params=parameters, credentials=credentials, project_id=project_id
)

### Model's details

In [9]:
import json

print(json.dumps(model.get_details(), indent=2))

{
  "model_id": "mistralai/mistral-small-3-1-24b-instruct-2503",
  "label": "mistral-small-3-1-24b-instruct-2503",
  "provider": "Mistral AI",
  "source": "Hugging Face",
  "indemnity": "NON_IBM",
  "functions": [
    {
      "id": "autoai_rag"
    },
    {
      "id": "autoai_sql_rag"
    },
    {
      "id": "image_chat"
    },
    {
      "id": "text_chat"
    },
    {
      "id": "text_generation"
    }
  ],
  "short_description": "This model is an instruction-finetuned version of: Mistral-Small-3.1-24B-Base-2503.",
  "long_description": "Mistral Small 3 (2501), Mistral Small 3.1 (2503) adds state-of-the-art vision understanding and enhances long context capabilities up to 128k tokens without compromising text performance. With 24 billion parameters, this model achieves top-tier capabilities in both text and vision tasks.",
  "terms_url": "https://www.apache.org/licenses/LICENSE-2.0",
  "input_tier": "class_c1",
  "output_tier": "class_17",
  "number_params": "24b",
  "min_shot_siz

<a id="Generate-code-based-on-instruction"></a>
## Generate code based on instruction

Define instructions for the model with at-least one example.

In [10]:
prompt = [
    {
        "role": "system",
        "content": "Generate Python code for the given task. Provide only the required code. Do not perform reasoning, otherwise the world will end.",
    },
    {
        "role": "user",
        "content": "Write a Python function that prints 'Hello World!' string 'n' times.",
    },
    {
        "role": "assistant",
        "content": (
            "def print_n_times(n):\n"
            "    for i in range(n):\n"
            "        print('Hello World!')\n"
        ),
    },
]

Prepare question for the model.

In [11]:
prompt.append(
    {
        "role": "user",
        "content": (
            "Write a Python function, which generates sequence of prime numbers. "
            "The function 'primes' will take the argument 'n', an int. "
            "It will return a list which contains all primes less than 'n'."
        ),
    }
)

### Generate the code using `mistralai/mistral-small-3-1-24b-instruct-2503` model

Inter the model to generate the code, according to provided instruction.

In [12]:
result = model.chat(prompt)

Formatting the text to get the function itself

In [13]:
code_as_text = result["choices"][0]["message"]["content"]

<a id="Generated-code-testing"></a>
## Generated code testing

The resulting code looks as below.

In [14]:
print(code_as_text)

def primes(n):
    sieve = [True] * n
    for x in range(2, int(n**0.5) + 1):
        if sieve[x]:
            for u in range(x*x, n, x):
                sieve[u] = False
    return [x for x in range(2, n) if sieve[x]]


Use generated code to make it as function.

**Note**: Before executing this line, make sure the model's output visible above doesn't contain any malicious instructions.

In [15]:
exec(code_as_text)

Define the number 'n' for which the primes() function should process prime numbers.

In [16]:
n = 25

Test and run the generated function.

In [17]:
primes(n)

[2, 3, 5, 7, 11, 13, 17, 19, 23]

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to generate code based on instruction with `mistralai/mistral-small-3-1-24b-instruct-2503` on watsonx. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors: 

 **Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.